# 🧪 Lab 01: Pipeline RAG (Retrieval-Augmented Generation) do Zero
**Módulo:** `03-genai-llms / rag-applications`

## Objetivos
1. Implementar a etapa de **Chunking** (fatiamento de documentos longos).
2. Construir o fluxo de **Recuperação por Relevância (Retrieval)**.
3. Montar o **Prompt Aumentado (Augmented Generation)** com contexto injetado para o LLM.

In [1]:
import numpy as np

# 1. Documento de Origem (Base de Conhecimento Externa)
documento_tecnico = """
O sistema OrionProv é uma engine de provisionamento automatizado de serviços de telecomunicações.
Ele utiliza filas assíncronas para processar ordens de serviço de banda larga e fibra óptica.
A arquitetura é baseada no padrão Saga para garantir a consistência das transações distribuídas.
Em caso de falha no equipamento do cliente, o fluxo aciona automaticamente ações de compensação.
O tempo médio de resposta para ativação de um novo assinante é inferior a 15 segundos.
"""

## 1. Etapa de Chunking (Fatiamento de Texto)

In [2]:
def chunk_text(text, chunk_size=2, overlap=1):
    sentences = [s.strip() for s in text.strip().split('.') if s.strip()]
    chunks = []
    
    for i in range(0, len(sentences), chunk_size - overlap):
        chunk = ". ".join(sentences[i:i + chunk_size]) + "."
        chunks.append(chunk)
        if i + chunk_size >= len(sentences):
            break
    return chunks

chunks = chunk_text(documento_tecnico, chunk_size=2, overlap=1)
print(f"Documento dividido em {len(chunks)} chunks:\n")
for idx, c in enumerate(chunks):
    print(f"Chunk {idx+1}: {c}")

Documento dividido em 4 chunks:

Chunk 1: O sistema OrionProv é uma engine de provisionamento automatizado de serviços de telecomunicações. Ele utiliza filas assíncronas para processar ordens de serviço de banda larga e fibra óptica.
Chunk 2: Ele utiliza filas assíncronas para processar ordens de serviço de banda larga e fibra óptica. A arquitetura é baseada no padrão Saga para garantir a consistência das transações distribuídas.
Chunk 3: A arquitetura é baseada no padrão Saga para garantir a consistência das transações distribuídas. Em caso de falha no equipamento do cliente, o fluxo aciona automaticamente ações de compensação.
Chunk 4: Em caso de falha no equipamento do cliente, o fluxo aciona automaticamente ações de compensação. O tempo médio de resposta para ativação de um novo assinante é inferior a 15 segundos.


## 2. Etapa de Recuperação (Retrieval)

In [3]:
def mock_embedding(text):
    np.random.seed(abs(hash(text)) % (2**32))
    v = np.random.randn(16)
    return v / np.linalg.norm(v)

# Indexação dos Chunks
vector_index = [{"chunk": c, "vector": mock_embedding(c)} for c in chunks]

def retrieve_context(query, top_k=2):
    query_vec = mock_embedding(query)
    
    for item in vector_index:
        dot = np.dot(query_vec, item["vector"])
        norm = np.linalg.norm(query_vec) * np.linalg.norm(item["vector"])
        item["score"] = dot / norm
        
    sorted_items = sorted(vector_index, key=lambda x: x["score"], reverse=True)
    return sorted_items[:top_k]

query_user = "Qual o tempo de ativação de um assinante no OrionProv?"
retrieved_chunks = retrieve_context(query_user, top_k=2)

print(f"❓ Pergunta do Usuário: {query_user}\n")
print("📌 Chunks Recuperados do Banco Vetorial:")
for item in retrieved_chunks:
    print(f"- [Score: {item['score']:.4f}] {item['chunk']}")

❓ Pergunta do Usuário: Qual o tempo de ativação de um assinante no OrionProv?

📌 Chunks Recuperados do Banco Vetorial:
- [Score: 0.2401] A arquitetura é baseada no padrão Saga para garantir a consistência das transações distribuídas. Em caso de falha no equipamento do cliente, o fluxo aciona automaticamente ações de compensação.
- [Score: -0.0528] Em caso de falha no equipamento do cliente, o fluxo aciona automaticamente ações de compensação. O tempo médio de resposta para ativação de um novo assinante é inferior a 15 segundos.


## 3. Etapa de Geração Aumentada (Augmented Prompt Construction)

In [4]:
def build_rag_prompt(query, retrieved_docs):
    contexto_str = "\n".join([f"• {doc['chunk']}" for doc in retrieved_docs])
    
    prompt_final = f"""
Você é um assistente técnico especializado. Responda à pergunta do usuário baseando-se EXCLUSIVAMENTE nas informações fornecidas no contexto abaixo.

--- CONTEXTO RECUPERADO ---
{contexto_str}
---------------------------

PERGUNTA: {query}

RESPOSTA:
"""
    return prompt_final

augmented_prompt = build_rag_prompt(query_user, retrieved_chunks)

print("=== PROMPT FINAL ENVIADO AO LLM (RAG) ===")
print(augmented_prompt)

=== PROMPT FINAL ENVIADO AO LLM (RAG) ===

Você é um assistente técnico especializado. Responda à pergunta do usuário baseando-se EXCLUSIVAMENTE nas informações fornecidas no contexto abaixo.

--- CONTEXTO RECUPERADO ---
• A arquitetura é baseada no padrão Saga para garantir a consistência das transações distribuídas. Em caso de falha no equipamento do cliente, o fluxo aciona automaticamente ações de compensação.
• Em caso de falha no equipamento do cliente, o fluxo aciona automaticamente ações de compensação. O tempo médio de resposta para ativação de um novo assinante é inferior a 15 segundos.
---------------------------

PERGUNTA: Qual o tempo de ativação de um assinante no OrionProv?

RESPOSTA:

